# 旧CAの信頼を外す


## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*


# 対象クラスタの設定


In [ ]:
target_group = 'hadoop_all_testcluster'
target_group


In [ ]:
import os
import yaml

with open('group_vars/' + target_group) as stream:
    configuration = yaml.safe_load(stream)


HTTPSで確認するサービスのホスト名とポートを指定する。リスニングポートと現在の設定を確認する。


In [ ]:
!ansible {target_group} --become -a 'ss -lntp'


In [ ]:
https_endpoints = [
    ('test01', 8481), ('test02', 8481),
    ('test01', 50470), ('test02', 50470),
    ('test01', 50475), ('test02', 50475),
    ('test01', 8088), ('test02', 8088),
    ('test01', 8044), ('test02', 8044),
    ('test01', 19890), ('test01', 8190),
]
https_endpoints


In [ ]:
pki_dir = os.path.expanduser('~/.local/share/hadoop-pki/replacement')
pki_dir


In [ ]:
ca_key_file = os.path.join(pki_dir, 'ca.key')
ca_certificate_file = os.path.join(pki_dir, 'ca.crt')
truststore_file = os.path.join(pki_dir, 'truststore.jks')
(ca_key_file, ca_certificate_file, truststore_file)


In [ ]:
os.environ['HADOOP_TRUSTSTORE_PASSWORD'] = configuration['https_truststore_pass']


# Playbookの準備


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
work_dir


In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1


In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir


In [ ]:
https_keystore_path = configuration['https_keystore_path']
https_truststore_path = configuration['https_truststore_path']
ca_bundle_path = '/etc/hadoop/conf/ca-certificates.pem'
(https_keystore_path, https_truststore_path, ca_bundle_path)


In [ ]:
next_ca_path = '/etc/hadoop/conf/ca-next.crt'
next_ca_path


# 旧CAの信頼を外す

すべての接続先が新CAだけで検証できることを確認する。旧CAで発行した証明書を使う接続先が残っている場合は、切り替えを済ませてから進む。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {next_ca_path} -connect {hostname}:{port} < /dev/null'


In [ ]:
!keytool -delete -alias old-root-ca -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -list -v -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!ansible {target_group} --become -m copy -a 'src={truststore_file} dest={https_truststore_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_certificate_file} dest={ca_bundle_path} owner=root group=root mode=0444'


In [ ]:
!ansible-playbook {playbook_dir}/restart_https_services.yml -l {target_group}


# 更新後の確認

新CAだけを含む設定でTLS接続を確認する。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {ca_bundle_path} -connect {hostname}:{port} < /dev/null'


In [ ]:
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs dfsadmin -report'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs fsck /'


In [ ]:
!ansible {target_group} --become -m file -a 'path={next_ca_path} state=absent'
!rm -fr {work_dir}


In [ ]:
del os.environ['HADOOP_TRUSTSTORE_PASSWORD']
